In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler

In [2]:
df = pd.read_csv("../dataset/hmnist_28_28_RGB.csv")
X = df.iloc[:,:-1]
Y = df.iloc[:,-1]

# Initialize RandomOverSampler
oversample = RandomOverSampler(random_state=42)

# Apply oversampling to balance the dataset
X, Y = oversample.fit_resample(X, Y)

# Convert to numpy arrays
X = X.to_numpy()
Y = Y.to_numpy()

X = X.reshape(-1, 28, 28, 3) #Reshape to original image dimensions (N, 28, 28, 3)


X = np.transpose(X, (0, 3, 1, 2))
X = X / 255.0

# Convert to PyTorch Tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(Y, dtype=torch.long) # In PyTorch, labels must be Long type

# Simple Split (80% train, 20% test) instead of K-Fold
# We use a single split to train one final model for verification
X_train, X_test, y_train, y_test = train_test_split(X_tensor, y_tensor, test_size=0.2, random_state=42)

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [3]:
class SkinCancerCNN(nn.Module):
    def __init__(self):
        super(SkinCancerCNN, self).__init__()
        
        self.features = nn.Sequential(
            # First Convolutional Block
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),              
            nn.MaxPool2d(2, 2),     # Reduces size to 14x14
           
            # Second Convolutional Block
           
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),     # Reduces size to 7x7
            
    
            nn.Flatten()
        )
        
        self.classifier = nn.Sequential(
            # Reduced Dense layers
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.2),        
            nn.Linear(128, 7)       # 7 Output classes
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Initialize model  
model = SkinCancerCNN()

In [ ]:
# Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Learning Rate Scheduler 
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

train_losses = []
train_accs = []
val_accs = []

EPOCHS = 15 

for epoch in range(EPOCHS):
    model.train() 
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        # Zero gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    # Validation phase (at the end of each epoch)
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    
    val_acc = 100 * val_correct / val_total
    
    scheduler.step(val_acc)
    
    train_acc = 100 * correct / total
    train_losses.append(running_loss / len(train_loader))
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")


In [21]:
torch.save(model.state_dict(), 'skin_model.pth')
print("\n[SUCCESS] Model saved as 'skin_model.pth'")

# Save data for verification 
# (We only save 20 images that the network classifies CORRECTLY)
model.eval()
images_to_verify = []
labels_to_verify = []
count = 0

print("Extracting images for verification...")
with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        
        # Filter: keep only if prediction matches label
        correct_mask = (predicted == labels)
        
        if correct_mask.any():
            # Convert to NumPy for saving
            valid_imgs = inputs[correct_mask].numpy()
            valid_lbls = labels[correct_mask].numpy()
            
            for img, lbl in zip(valid_imgs, valid_lbls):
                images_to_verify.append(img)
                labels_to_verify.append(lbl)
                count += 1
                if count >= 20: # We only need 20 examples
                    break
        if count >= 20:
            break

# Save as .npy files for Alpha-Beta-CROWN and Auto-LiRPA
X_verify = np.array(images_to_verify)
Y_verify = np.array(labels_to_verify)

np.save('data_X.npy', X_verify)
np.save('data_Y.npy', Y_verify)

print(f"[SUCCESS] Verification dataset ({count} images) saved as 'data_X.npy' and 'data_Y.npy'")



[SUCCESS] Model saved as 'skin_model.pth'
Extracting images for verification...
[SUCCESS] Verification dataset (20 images) saved as 'data_X.npy' and 'data_Y.npy'


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Pérdida y exactitud por época
plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(train_losses, marker='o')
plt.title('Pérdida de entrenamiento')
plt.xlabel('Época')
plt.ylabel('Loss')

plt.subplot(1,2,2)
plt.plot(train_accs, marker='o', label='Train')
plt.plot(val_accs, marker='o', label='Val')
plt.title('Exactitud')
plt.xlabel('Época')
plt.ylabel('Accuracy (%)')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Obtener predicciones sobre el set de test
model.eval()
y_true = []
y_pred = []
with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.numpy().tolist())
        y_pred.extend(preds.numpy().tolist())

# Matriz de confusión
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusión')
plt.xlabel('Predicha')
plt.ylabel('Verdadera')
plt.show()

# Reporte de clasificación
print(classification_report(y_true, y_pred, digits=4))

# Mostrar algunas imágenes de test con predicción vs etiqueta
def imshow_tensor(img):
    img = img.numpy()
    img = np.transpose(img, (1,2,0))
    plt.imshow(img)
    plt.axis('off')

# Tomar las primeras 9 imágenes del test_loader
examples = 9
fig = plt.figure(figsize=(9,9))
idx = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        for i in range(inputs.size(0)):
            if idx >= examples:
                break
            idx += 1
            ax = fig.add_subplot(3,3,idx)
            imshow_tensor(inputs[i])
            _, pred = torch.max(model(inputs[i].unsqueeze(0)), 1)
            ax.set_title(f"P:{pred.item()} / T:{labels[i].item()}")
        if idx >= examples:
            break
plt.tight_layout()
plt.show()